# `wrap_points_to_box()`

The grid utility `nematics3d.wrap_points_to_box()` maps three-dimensional points into the principal periodic box. Along each periodic axis of length $L$, coordinates are wrapped into the interval $[0, L)$; axes marked with `np.inf` are left non-periodic.

When a grid transform or offset is supplied, wrapping is performed in lattice coordinates and the result is converted back to physical coordinates.


## Setup

**For readers who are only interested in the tutorial, this section can be safely skipped.** The following cell imports `NumPy` and `Nematics3D`.


In [ ]:
import numpy as np
import nematics3d as n3d


## Minimal example

For a cubic periodic box of length 10, coordinates outside the principal box are reduced modulo 10.


In [ ]:
points = np.array([[12.0, -3.0, 7.0], [23.0, 8.0, 15.0]])
wrapped = n3d.wrap_points_to_box(points, box_size_periodic=10.0)
print(wrapped)


## Inputs and outputs

The public signature is:

```python
wrap_points_to_box(
    points,
    box_size_periodic=np.inf,
    transform=GRID_TRANSFORM_IDENTITY,
    offset=None,
)
```

`points` may be one point with shape `(3,)`, a collection with shape `(N, 3)`, or an empty collection. The returned array preserves the distinction between a single point `(3,)` and a collection `(N, 3)` and is independent of the input.

`box_size_periodic` may be one positive box length shared by all axes or three values for $x$, $y$, and $z$. Use `np.inf` for a non-periodic axis. For example, `[10.0, 20.0, np.inf]` wraps only $x$ and $y$.


## Mixed periodic and non-periodic axes

Only axes with finite box lengths are wrapped.


In [ ]:
point = np.array([12.0, -3.0, 27.0])
wrapped = n3d.wrap_points_to_box(point, [10.0, 20.0, np.inf])
print(wrapped)


## Grid transforms and offsets

If `transform` or `offset` is provided, `points` are interpreted as physical coordinates. `Nematics3D` first converts them to lattice coordinates, applies periodic wrapping there, and then converts the wrapped points back to physical coordinates.

In the following example, the lattice point `(12, -1, 2)` is first converted to physical coordinates and then wrapped. The wrapped lattice point is `(2, 9, 2)`.


In [ ]:
transform = np.array(
    [[0.0, 2.0, 0.0], [-3.0, 0.0, 0.0], [0.0, 0.0, 4.0]]
)
offset = np.array([5.0, 6.0, 7.0])

point_lattice = np.array([12.0, -1.0, 2.0])
point_physical = n3d.apply_linear_transform(point_lattice, transform, offset)
wrapped_physical = n3d.wrap_points_to_box(
    point_physical,
    box_size_periodic=10.0,
    transform=transform,
    offset=offset,
)
wrapped_lattice = n3d.apply_linear_transform(
    wrapped_physical, transform, offset, is_inv=True
)
print(wrapped_lattice)


## Important distinction

`wrap_points_to_box()` treats every point independently. It is appropriate when positions need to be represented inside the principal periodic box. It does **not** reconstruct a continuous trajectory across periodic boundaries; use `nematics3d.unwrap_trajectory()` for that purpose.
